# 第 12 章：MoE 融合算子和性能分析 — 章节实践与测试

## 小节概述

本小节包含两部分内容：

1. **综合编程实践**（简单 / 中等 / 困难各一道）：基于 12.02 工程完成新用例扩展、
   W 数据复用的 UB 驻留优化、多核共写 cache line 约束的复现实验；
2. **知识测验**：覆盖融合判据、流量估算、Top-K 选型、多核切分与 cache line 约束。

完成所有题目后，可运行最后的 cell 查看参考答案。

## 一、综合编程实践题

### 实践题 1（简单）：新增形状的测试用例

12.02 使用了 8 组固定用例。请在 `tools/gen_test_data.py` 的 `CASES` 列表中新增一个
用例 **`case_64_256_8_2`（N=64, D=256, E=8, K=2）**，重新生成数据并运行回归。

**任务**：

1. 修改 `tools/gen_test_data.py`（新增一行即可），运行 `python3 tools/gen_test_data.py`；
2. 运行 `bash src/custom_op/test/run.sh data/case_64_256_8_2`；
3. 根据 tiling 输出回答：该用例的 `blockDim` 与 `rowsPerCore` 是多少？为什么？
   （提示：K=2 时 `R_align = 64/gcd(4, 64) = 16` 行）

**验证标准**：新用例回归 `PASS`，且能正确解释切分参数。

### 实践题 2（中等）：W_gate 的 UB 驻留优化

12.02 的纯标量实现中，每一行都要对 W_gate 做 D·E 次标量 GM 读
（依赖 L2 缓存复用）。请将其改为**每核一次性驻留**：

**任务**：

1. 在 `KernelMoeRouterFused` 中新增成员数组 `float wT[kMaxE][kMaxD]`（转置后的
   W：`wT[e][d] = w_gate[d][e]`），在 `Process()` 的行循环之前用标量读填充一次；
2. 行循环内的点积改为从 `wT` 读取（UB 数组，不再访问 GM）；
3. 重新编译并运行 `bash tools/run_all.sh`（必须仍为 8/8 PASS）；
4. 用 `--bench` 对比 `case_4096_2048_8_2` 改造前后的耗时，解释变化。

**关键约束（先推导再动手）**：910B 单核栈帧上限 32KB（第 8 章已验证的约束）。
`float wT[kMaxE][kMaxD]` 需要 `E·D·4B` 字节——请推导本算子可驻留的最大 `D·E`
规模，并说明 `case_4096_2048_8_2`（D·E = 16384 元素 = 64KB）能否整体驻留；
不能整体驻留时给出分块驻留方案（按 d 维分块，块大小 ≤ 32KB）。

**验证标准**：回归全 PASS + 给出前后耗时对比与栈帧约束推导。

### 实践题 3（困难）：多核共写 cache line 丢写的复现与修复

12.02 步骤 4 指出：跨步行进切分（`n = coreId; n += blockDim`）会让多核并发写
同一条 64B cache line，导致**非确定性**丢写。请设计实验复现并闭环：

**任务**：

1. 将 `op_kernel/moe_router_fused.cpp` 的 `Process()` 临时改回跨步行进切分
   （`for (n = coreId; n < N; n += blockDim)`），并把 host 侧切分改回
   `blockDim = min(coreNum, N)`（其余不动），重新编译；
2. 对 `case_128_512_16_2` 连续运行 3 次（可用 `test/main.cpp --dump` 导出输出），
   记录每次的 `idx match / wt fail` 数值，验证其**非确定性**（每轮错法不同）；
3. 结合 `images/moe_tiling.svg` 解释：为什么该用例下所有核必然共写同一批 line？
   （提示：idx 总共 128×2×4B = 1KB，跨步行进时每条 64B line 内的 8 行来自不同核）
4. 恢复连续块切分，重新编译并确认 8/8 PASS 恢复（**务必恢复，避免污染后续实验**）。

**选做扩展**：尝试把点积改为 `DataCopy` 批量读 + 向量 `Mul`/`Add`（参考第 8 章
向量化实践题），记录 `aiv_vec_ratio` 变化与遇到的环境问题（若受环境限制无法
向量化到 PASS，记录现象与结论同样视为完成）。

**验证标准**：复现非确定性丢写的 3 组数据 + 原因解释 + 恢复后回归全 PASS。

In [ ]:
# ===== 实践题入口 =====
# 三道实践题均在 12.02 工程（src/custom_op/）上修改-编译-验证：
#   1) 修改源码后重新编译：   source $ASCEND_HOME_PATH/set_env.sh && cd src/custom_op && bash build.sh
#   2) 单用例验证：           bash src/custom_op/test/run.sh data/case_128_512_16_2
#   3) 全部 8 用例回归：       bash tools/run_all.sh
#   4) 性能对比：             bash src/custom_op/test/run.sh data/<case> --bench <iters>
#
# 本 cell 检查工程与数据是否就绪：
import os
for p in ['src/custom_op/op_kernel/moe_router_fused.cpp',
          'src/custom_op/op_host/moe_router_fused.cpp',
          'tools/gen_test_data.py', 'tools/run_all.sh']:
    print(('OK  ' if os.path.exists(p) else '缺失'), p)
if not os.path.isdir('data'):
    print('提示：data/ 不存在，请先运行 python3 tools/gen_test_data.py')
else:
    print(f'data/ 用例数: {len([d for d in os.listdir("data") if d.startswith("case_")])}')

## 二、选择题

### 题目 1：MoE Router 融合后，相比 4 算子序列消除的 GM 流量来自哪里？

In [ ]:
# 请将 answer1 修改为你的答案（A/B/C/D）
# A. 输入 x 的读取流量
# B. 中间张量 scores / gate_scores / topk_scores 的写与读（16·N·E + 8·N·K 字节）
# C. 输出 topk_idx 的写流量
# D. W_gate 的读取流量
answer1 = ''

### 题目 2：本算子的 Top-K（K ≤ E ≤ 32）采用 K 轮 "取最大 + 掩蔽"，其算法类别与复杂度是？

In [ ]:
# 请将 answer2 修改为你的答案（A/B/C/D）
# A. 排序类算法，O(E·log E)
# B. 选择类算法（selection），O(K·E)
# C. 哈希类算法，O(E)
# D. 归并类算法，O(K·log K)
answer2 = ''

### 题目 3：910B 上多核标量 GM 写非确定性丢失的根本原因是？

In [ ]:
# 请将 answer3 修改为你的答案（A/B/C/D）
# A. kernel 寄存器不足
# B. 多核并发写同一条 64B L2 cache line，核间无写一致性
# C. UB 容量不足
# D. Tiling 计算错误导致越界
answer3 = ''

### 题目 4：K=2 时，切分行对齐单位 R_align = 64/gcd(2K,64) 等于多少行？

In [ ]:
# 请将 answer4 修改为你的答案（A/B/C/D）
# A. 4 行
# B. 8 行
# C. 16 行
# D. 32 行
answer4 = ''

### 题目 5：本算子接口采用 FP16 而非 BF16 的直接原因是？

In [ ]:
# 请将 answer5 修改为你的答案（A/B/C/D）
# A. BF16 精度不足
# B. 本环境向量 Cast 不支持 BF16↔FP32（开发历程中的路线约束，数据与参考实现保持一致）
# C. FP16 动态范围更大
# D. BF16 无法存储在 GM
answer5 = ''

## 三、填空题

### 题目 6：910B 的 L2 cache line 大小为 ______ 字节（`kernel_utils_constants.h` 中 `CACHE_LINE_SIZE`）。

In [ ]:
answer6 = ''  # 请填写你的答案

### 题目 7：算子融合的可行性判据是：子图中间张量（按行分块后）容量 ______ 单核 UB 容量，且切分维度上无 ______ 依赖。

In [ ]:
answer7 = ''  # 请填写你的答案（两个空，用逗号分隔）

### 题目 8：融合算子将 Router 路径的 kernel 发射次数从 ______ 次合并为 ______ 次。

In [ ]:
answer8 = ''  # 请填写你的答案（两个空，用逗号分隔）

## 四、查看参考答案

完成所有题目后，运行以下 cell 查看参考答案文件清单与摘要
（完整答案位于 `answer/` 目录：`chapter_test_answer.md` 为知识测验答案与解析，
`practice_answer.md` 为三道实践题的参考实现）：

In [ ]:
# 查看参考答案摘要
import subprocess
print('=== answer/README.md ===')
subprocess.run(['cat', 'answer/README.md'])
print()
print('=== answer/chapter_test_answer.md（前 40 行）===')
subprocess.run(['sed', '-n', '1,40p', 'answer/chapter_test_answer.md'])